# 21 — Comparing Multiple Studies

Load the same sensor channel from several bearings at once and overlay their degradation curves on one interactive figure.

**Dataset**: XJTU-SY — all 5 bearings in the 35 Hz / 12 kN condition (Bearing 2_1 … 2_5)  
**API**: `wrapper.compare_studies()` · `plotter.plot_multi_lifecycle()`
Run the import and dataset-loading cells first. Each following example selects what it needs independently; change its study, assay, or run to explore your data.


## Imports and dataset path


In [ ]:
from pathlib import Path

from isa_phm import ISAWrapper
from isa_phm.plotter import ISAPlotter
from bokeh.io import output_notebook
from bokeh.plotting import show as bokeh_show

output_notebook()

# Set this to your dataset's ISA-PHM JSON file.
ISA_JSON = Path("path/to/your-dataset.isa-phm.json").expanduser()
ISA_JSON = Path("path/to/your-dataset.isa-phm.json").expanduser()
DATA_ROOT = ISA_JSON.parent

print("ISA-JSON :", ISA_JSON)
print("DATA_ROOT:", DATA_ROOT)
print("Exists   :", ISA_JSON.exists())


## Load the wrapper


In [ ]:
wrapper = ISAWrapper(
    ISA_JSON,
    data_root=DATA_ROOT,
    strict_validation=False,
    cache_maxsize=20,
)


## Dataset overview


In [ ]:
print(f"{wrapper.investigation_overview().n_studies} studies in dataset")


## 1. Load lifecycle features for the first two studies at once

`compare_studies()` selects the same assay from each study and loads all lifecycle features in parallel.  
No manual for-loop needed — the result is a `dict[study_title, DataFrame]` ready for plotting.

Only the studies you pass are included.

In [ ]:
STUDY_INDICES = [1, 2]  # 1-based: first and second study

# assay_id=1 → first assay (horizontal accelerometer) of each study
lc_dict = wrapper.compare_studies(
    STUDY_INDICES,
    assay_id=1,
    file_type="raw",
    n_workers=8,
)

print(f"{len(lc_dict)} studies loaded:")
for name, lc in lc_dict.items():
    print(f"  {name}: {len(lc)} runs")

## 2. RMS lifecycle comparison

Overlay all five RMS curves. Click a legend entry to hide/show individual bearings.

In [ ]:
lc_dict = wrapper.compare_studies([1, 2], assay_id=1, file_type="raw", n_workers=8)
plotter = ISAPlotter()

fig = plotter.plot_multi_lifecycle(
    lc_dict,
    feature="rms",
    title="Study 1, 2, RMS Degradation",
)
bokeh_show(fig)

## 3. Kurtosis comparison

Kurtosis shows the timing of the impulsive fault onset — which bearing failed earliest?

In [ ]:
lc_dict = wrapper.compare_studies([1, 2], assay_id=1, file_type="raw", n_workers=8)
plotter = ISAPlotter()

fig = plotter.plot_multi_lifecycle(
    lc_dict,
    feature="kurtosis",
    title="Study 1, 2 — Kurtosis",
)
bokeh_show(fig)

## 4. Crest factor comparison

Crest factor often rises *before* RMS, making it useful for early detection.

In [ ]:
lc_dict = wrapper.compare_studies([1, 2], assay_id=1, file_type="raw", n_workers=8)
plotter = ISAPlotter()

fig = plotter.plot_multi_lifecycle(
    lc_dict,
    feature="crest_factor",
    title="Study 1, 2 — Crest Factor",
)
bokeh_show(fig)

## 5. Cross-condition comparison

`study.compare_with()` lets you start from one study and add others — including bearings from different operating conditions.

The base study (`self`) is always included.

In [ ]:
plotter = ISAPlotter()

study1 = wrapper.study(1)
study2 = wrapper.study(2)

cross_dict = study1.compare_with(
    [study2],
    assay_id=2,
    file_type="raw",
    n_workers=8,
)

fig = plotter.plot_multi_lifecycle(
    cross_dict,
    feature="crest_factor",
    title="Cross-condition crest_factor Degradation",
)
bokeh_show(fig)